In [10]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score
from PIL import Image
import timm

# --- CONFIG ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
IMG_SIZE = 224 # Faster iteration for ensemble
BATCH_SIZE = 32

# --- ENSEMBLE MODELS ---
class CommitteeNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Expert 1: Convolutional (Local Textures)
        self.expert_cnn = timm.create_model('convnext_tiny', pretrained=True, num_classes=5)
        # Expert 2: Transformer (Global Relationships)
        self.expert_vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=5)
        
    def forward(self, x):
        out_cnn = self.expert_cnn(x)
        out_vit = self.expert_vit(x)
        # Ensemble via soft-voting (averaging probabilities)
        return (out_cnn + out_vit) / 2

# --- DATASET ---
class AptosDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def train_and_evaluate():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CommitteeNet().to(device)
    
    # We use a standard CE with Label Smoothing for better Accuracy
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(AptosDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(AptosDataset('valid.csv', VAL_IMG_DIR, train_tf), batch_size=BATCH_SIZE)

    print("Training Heterogeneous Ensemble Committee...")
    best_acc = 0
    
    for epoch in range(10):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                preds.extend(torch.argmax(out, 1).cpu().numpy())
                targets.extend(labels.cpu().numpy())

        acc = accuracy_score(targets, preds)
        kappa = cohen_kappa_score(targets, preds, weights='quadratic')
        print(f"Epoch {epoch+1} | Accuracy: {acc:.4%}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), 'committee_model.pth')

    # FINAL DETAILED REPORT
    print("\n" + "="*50)
    print("      FINAL ENSEMBLE CLINICAL REPORT")
    print("="*50)
    model.load_state_dict(torch.load('committee_model.pth'))
    model.eval()
    
    f_preds, f_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            f_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
            f_targets.extend(labels.cpu().numpy())

    print(f"Accuracy: {accuracy_score(f_targets, f_preds):.4%}")
    print(f"Kappa:    {cohen_kappa_score(f_targets, f_preds, weights='quadratic'):.4f}")
    
    cm = confusion_matrix(f_targets, f_preds)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Prolif']
    for i in range(5):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = sum(cm.flatten()) - (tp + fn + fp)
        print(f"{classes[i]:<10} | Sens: {tp/(tp+fn):.4f} | Spec: {tn/(tn+fp):.4f}")

if __name__ == "__main__":
    train_and_evaluate()

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Training Heterogeneous Ensemble Committee...
Epoch 1 | Accuracy: 80.3279%
Epoch 2 | Accuracy: 83.6066%
Epoch 3 | Accuracy: 83.6066%
Epoch 4 | Accuracy: 85.2459%
Epoch 5 | Accuracy: 84.6995%
Epoch 6 | Accuracy: 86.3388%
Epoch 7 | Accuracy: 84.4262%
Epoch 8 | Accuracy: 85.7923%
Epoch 9 | Accuracy: 86.3388%
Epoch 10 | Accuracy: 85.7923%

      FINAL ENSEMBLE CLINICAL REPORT
Accuracy: 86.6120%
Kappa:    0.9110
No DR      | Sens: 0.9942 | Spec: 0.9845
Mild       | Sens: 0.6500 | Spec: 0.9816
Moderate   | Sens: 0.8654 | Spec: 0.9084
Severe     | Sens: 0.6818 | Spec: 0.9593
Prolif     | Sens: 0.5357 | Spec: 0.9941
